In [152]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6"
device = "cuda"

In [153]:
import numpy as np
import networkx as nx
from typing import List, Dict, Tuple, Set, Optional
from dataclasses import dataclass
from collections import defaultdict
import spacy
import os
import requests
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import re
import warnings
from pathlib import Path
import pandas as pd
import pickle

In [155]:
# Suppress warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# model = "gemma3:12b"
model = "gemma3:27b-it-qat"
# Configurazione Ollama
os.environ["OLLAMA_HOST"] = "http://localhost:11502"
os.environ["OLLAMA_MODEL_NAME"] = model
os.environ["OLLAMA_EMBEDDING_MODEL_NAME"] = "nomic-embed-text:latest"

In [156]:
dataset = 'mocheg'#hover  CAMBIARE PER CAMBIARE DATASET
WORKING_DIR = f"dataset_{dataset}"
DOCUMENTS_DIR = f"data/{dataset}"

dataset_name = f"{dataset}.csv"

CLAIMS_CSV = f"data/{dataset_name}" 

In [157]:
@dataclass
class Triple:
    subject: str
    relation: str
    object: str
    source_sentence: str = ""
    doc_id: int = -1
    sent_id: int = -1


@dataclass
class Community:
    id: int
    nodes: Set[str]
    edges: List[Triple]
    embedding: Optional[np.ndarray] = None
    modularity_score: float = 0.0


@dataclass
class SelectedEdge:
    parent: str
    relation: str
    child: str
    source_sentence: str
    score: float
    hop: int
    community_id: int
    doc_id: int = -1
    sent_id: int = -1

In [158]:
class OllamaClient:
    """Client per interagire con Ollama API"""
    
    def __init__(self, host: str = None, model_name: str = None, embedding_model: str = None):
        self.host = host or os.environ.get("OLLAMA_HOST", "http://localhost:11502")
        self.model_name = model_name or os.environ.get("OLLAMA_MODEL_NAME", "gemma3:27b-it-qat")
        self.embedding_model = embedding_model or os.environ.get("OLLAMA_EMBEDDING_MODEL_NAME", "nomic-embed-text:latest")
        
    def get_embedding(self, text: str) -> np.ndarray:
        """Ottieni embedding da Ollama"""
        url = f"{self.host}/api/embeddings"
        payload = {"model": self.embedding_model, "prompt": text}
        
        response = requests.post(url, json=payload, timeout=60)
        response.raise_for_status()
        embedding = response.json()["embedding"]
        return np.array(embedding, dtype=np.float32)
    
    def generate(self, prompt: str, temperature: float = 0.3, max_tokens: int = 200) -> str:
        """Genera testo usando Ollama con parametri del paper (Appendix C)"""
        url = f"{self.host}/api/generate"
        payload = {
            "model": self.model_name,
            "prompt": prompt,
            "temperature": temperature,
            "stream": False,
            "options": {
                "num_predict": max_tokens,
                "repeat_penalty": 1.1  # Paper Appendix C
            }
        }
        
        response = requests.post(url, json=payload, timeout=120)
        response.raise_for_status()
        return response.json()["response"]

In [159]:
class CoreferenceResolver:
    """
    Section 4.1.1: Coreference Resolution
    Paper: "We employ a state-of-the-art coreference resolution model by Lee et al. (2018),
    leveraging a deep learning approach based on SpanBERT"
    """
    
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        self.method = "rule_based"
        
        # Prova FastCoref
        try:
            from fastcoref import spacy_component
            self.nlp.add_pipe(
                "fastcoref",
                config={'model_architecture': 'LingMessCoref', 'model_path': 'biu-nlp/lingmess-coref'}
            )
            self.method = "fastcoref"
        except:
            pass
    
    def resolve(self, text: str) -> str:
        """Risolve le coreferenze nel testo"""
        if self.method == "fastcoref":
            try:
                doc = self.nlp(text)
                
                if hasattr(doc._, 'coref_clusters') and doc._.coref_clusters:
                    resolved_tokens = []
                    mention_to_main = {}
                    
                    for cluster in doc._.coref_clusters:
                        if len(cluster) > 0:
                            main_mention = cluster[0]
                            for mention in cluster:
                                mention_to_main[mention] = main_mention
                    
                    for token in doc:
                        replaced = False
                        for mention, main in mention_to_main.items():
                            if token.i >= mention.start and token.i < mention.end:
                                if token.i == mention.start:
                                    resolved_tokens.append(main.text)
                                replaced = True
                                break
                        
                        if not replaced:
                            resolved_tokens.append(token.text)
                    
                    return " ".join(resolved_tokens)
                
                return text
            except:
                return self._rule_based_resolve(text)
        else:
            return self._rule_based_resolve(text)
    
    def _rule_based_resolve(self, text: str) -> str:
        """Fallback rule-based per coreference"""
        doc = self.nlp(text)
        resolved_tokens = []
        last_subject = None
        
        for token in doc:
            if token.dep_ in ("nsubj", "nsubjpass") and token.pos_ in ("PROPN", "NOUN"):
                last_subject = token.text
                resolved_tokens.append(token.text)
            elif token.lower_ in ("he", "she", "it") and last_subject:
                resolved_tokens.append(last_subject)
            elif token.lower_ in ("his", "her", "its") and last_subject:
                resolved_tokens.append(f"{last_subject}'s")
            else:
                resolved_tokens.append(token.text_with_ws)
        
        return "".join(resolved_tokens)

In [160]:
class REBELRelationExtractor:
    """
    Section 4.1.2: Graph Construction
    Paper: "CommunityKG-RAG leverages the relationship extraction model, REBEL"
    """
    
    def __init__(self, device: str = "cuda" if torch.cuda.is_available() else "cpu"):
        self.device = device
        
        try:
            model_name = "Babelscape/rebel-large"
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
            self.model.eval()
            self.available = True
        except:
            self.available = False
            self.nlp = spacy.load("en_core_web_sm")
    
    def extract_relations(self, text: str) -> List[Triple]:
        """Estrae triple (subject, relation, object) dal testo"""
        if self.available:
            return self._extract_with_rebel(text)
        else:
            return self._extract_with_spacy(text)
    
    def _extract_with_rebel(self, text: str) -> List[Triple]:
        """Estrae relazioni usando REBEL"""
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(self.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_length=256,
                num_beams=5,
                num_return_sequences=1,
                early_stopping=True
            )
        
        decoded = self.tokenizer.decode(outputs[0], skip_special_tokens=False)
        triples = self._parse_rebel_output(decoded, text)
        return triples
    
    def _parse_rebel_output(self, rebel_output: str, source_text: str) -> List[Triple]:
        """Parse REBEL output"""
        triples = []
        triplet_pattern = r'<triplet>(.*?)(?=<triplet>|$)'
        triplets = re.findall(triplet_pattern, rebel_output, re.DOTALL)
        
        for triplet in triplets:
            try:
                parts = re.split(r'<subj>|<obj>', triplet)
                if len(parts) >= 3:
                    relation = parts[0].strip()
                    subject = parts[1].strip()
                    obj = parts[2].strip()
                    
                    if subject and relation and obj:
                        triples.append(Triple(
                            subject=subject,
                            relation=relation,
                            object=obj,
                            source_sentence=source_text
                        ))
            except:
                continue
        
        return triples
    
    def _extract_with_spacy(self, text: str) -> List[Triple]:
        """Fallback usando Spacy"""
        doc = self.nlp(text)
        triples = []
        
        entities = {ent.text: ent.label_ for ent in doc.ents}
        important_nouns = {token.text for token in doc 
                          if token.pos_ in ("PROPN", "NOUN") 
                          and not token.is_stop and len(token.text) > 2}
        all_entities = set(entities.keys()) | important_nouns
        
        for sent in doc.sents:
            root_verb = next((token for token in sent 
                            if token.dep_ == "ROOT" and token.pos_ == "VERB"), None)
            
            if root_verb:
                subjects = [" ".join([t.text for t in child.subtree])
                           for child in root_verb.children 
                           if child.dep_ in ("nsubj", "nsubjpass")]
                
                objects = []
                for child in root_verb.children:
                    if child.dep_ in ("dobj", "attr"):
                        objects.append(" ".join([t.text for t in child.subtree]))
                    elif child.dep_ == "prep":
                        for prep_child in child.children:
                            if prep_child.dep_ == "pobj":
                                objects.append(" ".join([t.text for t in prep_child.subtree]))
                
                relation = root_verb.lemma_
                for subj in subjects:
                    for obj in objects:
                        if subj.strip() and obj.strip():
                            triples.append(Triple(
                                subject=subj.strip(),
                                relation=relation,
                                object=obj.strip(),
                                source_sentence=text
                            ))
        
        entity_list = list(all_entities)
        if len(entity_list) >= 2:
            for i in range(min(len(entity_list), 5)):
                for j in range(i + 1, min(len(entity_list), 5)):
                    triples.append(Triple(
                        subject=entity_list[i],
                        relation="co_occurs_with",
                        object=entity_list[j],
                        source_sentence=text
                    ))
        
        return triples

In [161]:
class LouvainCommunityDetector:
    """Section 4.2: Community Detection"""
    
    def __init__(self):
        try:
            from community import community_louvain
            self.louvain = community_louvain
            self.method = "louvain"
        except ImportError:
            self.method = "greedy"
    
    def detect_communities(self, graph: nx.Graph) -> Dict[str, int]:
        """Rileva communities"""
        if self.method == "louvain":
            return self.louvain.best_partition(graph)
        else:
            communities_gen = nx.community.greedy_modularity_communities(graph)
            partition = {}
            for idx, comm in enumerate(communities_gen):
                for node in comm:
                    partition[node] = idx
            return partition
    
    def compute_modularity(self, graph: nx.Graph, partition: Dict[str, int]) -> float:
        """Calcola modularity"""
        if self.method == "louvain":
            return self.louvain.modularity(partition, graph)
        else:
            return nx.community.modularity(
                graph,
                [set(n for n, c in partition.items() if c == comm_id) 
                 for comm_id in set(partition.values())]
            )

In [167]:
from dataclasses import dataclass
from collections import defaultdict, deque
from typing import List, Dict, Tuple, Set, Optional, Any
import numpy as np
import networkx as nx
import re
import pickle
import torch





class CommunityKGRAG:
    def __init__(
        self,
        ollama_host: str = None,
        ollama_model: str = None,
        ollama_embedding_model: str = None,
        device: str = "cuda" if torch.cuda.is_available() else "cpu",
        verbose: bool = False
    ):
        self.device = device
        self.verbose = verbose

        self.ollama = OllamaClient(ollama_host, ollama_model, ollama_embedding_model)
        self.coref_resolver = CoreferenceResolver()
        self.relation_extractor = REBELRelationExtractor(device)
        self.community_detector = LouvainCommunityDetector()

        self.kg = nx.MultiDiGraph()
        self.communities: Dict[int, Community] = {}
        self.partition: Dict[str, int] = {}

        self.sentence_to_triples: Dict[str, List[Triple]] = defaultdict(list)
        self.node_to_sentences: Dict[str, Set[str]] = defaultdict(set)

        self.node_embeddings: Dict[str, np.ndarray] = {}
        self.sentence_embeddings: Dict[str, np.ndarray] = {}
        self.relation_embeddings: Dict[str, np.ndarray] = {}
        self.claim_embeddings: Dict[str, np.ndarray] = {}

    def _log(self, *args):
        if self.verbose:
            print(*args)

    def _normalize_text(self, text: str) -> str:
        text = re.sub(r"\s+", " ", str(text)).strip()
        return text

    def _cosine(self, a: np.ndarray, b: np.ndarray) -> float:
        if a is None or b is None:
            return 0.0
        na = np.linalg.norm(a)
        nb = np.linalg.norm(b)
        if na == 0 or nb == 0:
            return 0.0
        return float(np.dot(a, b) / (na * nb))

    def _get_embedding(self, text: str, cache: Optional[Dict[str, np.ndarray]] = None) -> np.ndarray:
        text = self._normalize_text(text)
        if cache is not None and text in cache:
            return cache[text]
        emb = self.ollama.get_embedding(text)
        emb = np.array(emb, dtype=np.float32)
        if cache is not None:
            cache[text] = emb
        return emb

    def _get_node_embedding(self, node: str) -> np.ndarray:
        node = self._normalize_text(node)
        if node not in self.node_embeddings:
            self.node_embeddings[node] = self._get_embedding(node)
        return self.node_embeddings[node]

    def _get_sentence_embedding(self, sentence: str) -> np.ndarray:
        sentence = self._normalize_text(sentence)
        if sentence not in self.sentence_embeddings:
            self.sentence_embeddings[sentence] = self._get_embedding(sentence)
        return self.sentence_embeddings[sentence]

    def _get_relation_embedding(self, relation: str) -> np.ndarray:
        relation = self._normalize_text(relation)
        if relation not in self.relation_embeddings:
            self.relation_embeddings[relation] = self._get_embedding(relation)
        return self.relation_embeddings[relation]

    def _get_claim_embedding(self, claim: str) -> np.ndarray:
        claim = self._normalize_text(claim)
        if claim not in self.claim_embeddings:
            self.claim_embeddings[claim] = self._get_embedding(claim)
        return self.claim_embeddings[claim]

    def _sentence_key(self, doc_id: int, sent_id: int) -> str:
        return f"d{doc_id}_s{sent_id}"

    def build_knowledge_graph(self, documents: List[str]):
        for doc_id, document in enumerate(documents):
            self._log("--Document number--", doc_id)
            resolved_text = self.coref_resolver.resolve(document)
            sentences = [
                self._normalize_text(s)
                for s in re.split(r'(?<=[.!?])\s+', resolved_text)
                if self._normalize_text(s)
            ]

            for sent_id, sentence in enumerate(sentences):
                triples = self.relation_extractor.extract_relations(sentence)
                clean_triples = []

                for triple in triples:
                    subj = self._normalize_text(triple.subject)
                    rel = self._normalize_text(triple.relation)
                    obj = self._normalize_text(triple.object)

                    if not subj or not rel or not obj:
                        continue
                    if len(subj) < 2 or len(obj) < 2 or len(rel) < 2:
                        continue
                    if subj == obj:
                        continue

                    t = Triple(
                        subject=subj,
                        relation=rel,
                        object=obj,
                        source_sentence=sentence,
                        doc_id=doc_id,
                        sent_id=sent_id
                    )
                    clean_triples.append(t)

                if not clean_triples:
                    continue

                sent_key = self._sentence_key(doc_id, sent_id)
                self.sentence_to_triples[sent_key].extend(clean_triples)

                for triple in clean_triples:
                    self.kg.add_node(triple.subject)
                    self.kg.add_node(triple.object)

                    self.node_to_sentences[triple.subject].add(triple.source_sentence)
                    self.node_to_sentences[triple.object].add(triple.source_sentence)

                    edge_key = f"{triple.doc_id}_{triple.sent_id}_{len(self.sentence_to_triples[sent_key])}"
                    self.kg.add_edge(
                        triple.subject,
                        triple.object,
                        key=edge_key,
                        relation=triple.relation,
                        source_sentence=triple.source_sentence,
                        doc_id=triple.doc_id,
                        sent_id=triple.sent_id
                    )

        self._compute_node_embeddings()

    def _compute_node_embeddings(self):
        for node in self.kg.nodes():
            emb = self._get_node_embedding(node)
            self.kg.nodes[node]["embedding"] = emb

    def _to_undirected_projection(self) -> nx.Graph:
        g = nx.Graph()
        for node in self.kg.nodes():
            g.add_node(node)

        for u, v, data in self.kg.edges(data=True):
            if g.has_edge(u, v):
                g[u][v]["weight"] += 1
            else:
                g.add_edge(u, v, weight=1)
        return g

    def detect_communities(self):
        undirected_kg = self._to_undirected_projection()
        partition = self.community_detector.detect_communities(undirected_kg)
        modularity = self.community_detector.compute_modularity(undirected_kg, partition)
        self.partition = partition

        communities_dict = defaultdict(set)
        for node, comm_id in partition.items():
            communities_dict[comm_id].add(node)

        self.communities = {}
        for comm_id, nodes in communities_dict.items():
            community_edges = []
            for u, v, key, data in self.kg.edges(keys=True, data=True):
                if u in nodes and v in nodes:
                    community_edges.append(
                        Triple(
                            subject=u,
                            relation=data.get("relation", "related"),
                            object=v,
                            source_sentence=data.get("source_sentence", ""),
                            doc_id=data.get("doc_id", -1),
                            sent_id=data.get("sent_id", -1)
                        )
                    )

            node_embeddings = [self.kg.nodes[node]["embedding"] for node in nodes if "embedding" in self.kg.nodes[node]]
            community_embedding = np.mean(node_embeddings, axis=0) if node_embeddings else None

            self.communities[comm_id] = Community(
                id=comm_id,
                nodes=nodes,
                edges=community_edges,
                embedding=community_embedding,
                modularity_score=modularity
            )

    def select_top_communities(self, claim: str, delta: float = 0.25) -> List[int]:
        if not self.communities:
            return []

        claim_embedding = self._get_claim_embedding(claim)
        scores = {}
        for comm_id, comm in self.communities.items():
            if comm.embedding is None:
                continue
            scores[comm_id] = self._cosine(claim_embedding, comm.embedding)

        if not scores:
            return []

        n = max(1, int(np.ceil(delta * len(scores))))
        top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]
        return [comm_id for comm_id, _ in top]

    def get_central_node(self, claim: str, community_id: int) -> Optional[str]:
        if community_id not in self.communities:
            return None

        claim_embedding = self._get_claim_embedding(claim)
        best_node = None
        best_score = -1e9

        for node in self.communities[community_id].nodes:
            node_embedding = self.kg.nodes[node].get("embedding")
            score = self._cosine(claim_embedding, node_embedding)
            if score > best_score:
                best_score = score
                best_node = node

        return best_node

    def _context_embedding(self, selected_nodes: Set[str]) -> Optional[np.ndarray]:
        if not selected_nodes:
            return None
        vectors = [self.kg.nodes[n]["embedding"] for n in selected_nodes if "embedding" in self.kg.nodes[n]]
        if not vectors:
            return None
        return np.mean(vectors, axis=0)




    def _score_candidate(
        self,
        claim_embedding: np.ndarray,
        relation: str,
        child_node: str,
        selected_nodes: Set[str],
        alpha: float = 0.45,
        beta: float = 0.35,
        gamma: float = 0.20
    ) -> float:
        child_emb = self.kg.nodes[child_node].get("embedding")
        rel_emb = self._get_relation_embedding(relation)
        ctx_emb = self._context_embedding(selected_nodes)
    
        sim_q_v = self._cosine(claim_embedding, child_emb)
        sim_v_s = self._cosine(child_emb, ctx_emb) if ctx_emb is not None else 0.0
        sim_r_q = self._cosine(rel_emb, claim_embedding)
    
        score = alpha * sim_q_v - beta * sim_v_s + gamma * sim_r_q
        return float(score)

    
    def prune_community_bfs(
        self,
        claim: str,
        community_id: int,
        tau: float = 0.15,
        top_k: int = 2,
        t_max: int = 3,
        budget: int = 12,
        alpha: float = 0.45,
        beta: float = 0.35,
        gamma: float = 0.20
    ) -> Dict[str, Any]:
        if community_id not in self.communities:
            return {
                "community_id": community_id,
                "center_node": None,
                "selected_nodes": set(),
                "selected_edges": []
            }
    
        community_nodes = self.communities[community_id].nodes
        center_node = self.get_central_node(claim, community_id)
        if center_node is None:
            return {
                "community_id": community_id,
                "center_node": None,
                "selected_nodes": set(),
                "selected_edges": []
            }
    
        claim_embedding = self._get_claim_embedding(claim)
        selected_nodes = {center_node}
        selected_edges = []
        frontier = deque([(center_node, 0)])
    
        while frontier and len(selected_nodes) < budget:
            current_node, hop = frontier.popleft()
            if hop >= t_max:
                continue
    
            candidates = []
            for _, child, key, data in self.kg.out_edges(current_node, keys=True, data=True):
                if child not in community_nodes or child in selected_nodes:
                    continue
    
                relation = data.get("relation", "related")
                score = self._score_candidate(
                    claim_embedding=claim_embedding,
                    relation=relation,
                    child_node=child,
                    selected_nodes=selected_nodes,
                    alpha=alpha,
                    beta=beta,
                    gamma=gamma
                )
    
                candidates.append({
                    "parent": current_node,
                    "child": child,
                    "relation": relation,
                    "source_sentence": data.get("source_sentence", ""),
                    "score": score,
                    "hop": hop + 1,
                    "doc_id": data.get("doc_id", -1),
                    "sent_id": data.get("sent_id", -1)
                })
    
            if not candidates:
                continue
    
            candidates = sorted(candidates, key=lambda x: x["score"], reverse=True)
    
            keep = [cand for cand in candidates if cand["score"] >= tau]
            forced_top = candidates[:top_k]
    
            keep_map = {}
            for cand in keep + forced_top:
                keep_map[(cand["parent"], cand["relation"], cand["child"], cand["doc_id"], cand["sent_id"])] = cand
    
            keep_final = sorted(keep_map.values(), key=lambda x: x["score"], reverse=True)
    
            for cand in keep_final:
                if len(selected_nodes) >= budget:
                    break
                if cand["child"] in selected_nodes:
                    continue
    
                selected_nodes.add(cand["child"])
                selected_edges.append(
                    SelectedEdge(
                        parent=cand["parent"],
                        relation=cand["relation"],
                        child=cand["child"],
                        source_sentence=cand["source_sentence"],
                        score=cand["score"],
                        hop=cand["hop"],
                        community_id=community_id,
                        doc_id=cand["doc_id"],
                        sent_id=cand["sent_id"]
                    )
                )
                frontier.append((cand["child"], hop + 1))
    
        return {
            "community_id": community_id,
            "center_node": center_node,
            "selected_nodes": selected_nodes,
            "selected_edges": selected_edges
        }
    
    
    def retrieve_evidence(
        self,
        claim: str,
        delta: float = 0.25,
        tau: float = 0.15,
        top_k: int = 2,
        t_max: int = 3,
        budget: int = 12,
        max_evidence: int = 12,
        alpha: float = 0.45,
        beta: float = 0.35,
        gamma: float = 0.20
    ) -> Dict[str, Any]:
        top_communities = self.select_top_communities(claim, delta=delta)
        all_selected_edges = []
        community_centers = {}
    
        for community_id in top_communities:
            result = self.prune_community_bfs(
                claim=claim,
                community_id=community_id,
                tau=tau,
                top_k=top_k,
                t_max=t_max,
                budget=budget,
                alpha=alpha,
                beta=beta,
                gamma=gamma
            )
            community_centers[community_id] = result["center_node"]
            all_selected_edges.extend(result["selected_edges"])
    
        all_selected_edges = sorted(all_selected_edges, key=lambda e: e.score, reverse=True)
        evidence_lines = self.linearize_subgraph_to_evidence(all_selected_edges, max_items=max_evidence)
    
        return {
            "top_communities": top_communities,
            "community_centers": community_centers,
            "selected_edges": all_selected_edges,
            "evidence_lines": evidence_lines
        }
    
    
    def verify_claim(
        self,
        claim: str,
        delta: float = 0.25,
        tau: float = 0.15,
        top_k: int = 2,
        t_max: int = 3,
        budget: int = 12,
        max_evidence: int = 12,
        alpha: float = 0.45,
        beta: float = 0.35,
        gamma: float = 0.20,
        use_llm: bool = True
    ) -> Dict[str, Any]:
        retrieval = self.retrieve_evidence(
            claim=claim,
            delta=delta,
            tau=tau,
            top_k=top_k,
            t_max=t_max,
            budget=budget,
            max_evidence=max_evidence,
            alpha=alpha,
            beta=beta,
            gamma=gamma
        )
    
        evidence_lines = retrieval["evidence_lines"]
    
        prediction = None
        llm_response = None
        if use_llm and evidence_lines:
            prediction, llm_response = self._verify_with_llm(claim, evidence_lines)
    
        return {
            "claim": claim,
            "prediction": prediction,
            "llm_response": llm_response,
            "evidence": evidence_lines,
            "top_communities": retrieval["top_communities"],
            "community_centers": retrieval["community_centers"],
            "selected_edges": retrieval["selected_edges"],
            "num_sentences": len(evidence_lines),
            "used_parameters": {
                "delta": delta,
                "tau": tau,
                "top_k": top_k,
                "t_max": t_max,
                "budget": budget,
                "max_evidence": max_evidence,
                "alpha": alpha,
                "beta": beta,
                "gamma": gamma,
                "use_llm": use_llm
            }
        }


    def _deduplicate_selected_edges(self, edges: List[SelectedEdge]) -> List[SelectedEdge]:
        seen = set()
        out = []
        for e in edges:
            key = (e.parent, e.relation, e.child, e.source_sentence, e.doc_id, e.sent_id)
            if key not in seen:
                seen.add(key)
                out.append(e)
        return out

    def linearize_subgraph_to_evidence(
        self,
        selected_edges: List[SelectedEdge],
        max_items: int = 12
    ) -> List[str]:
        selected_edges = self._deduplicate_selected_edges(selected_edges)
        selected_edges = sorted(selected_edges, key=lambda e: (e.hop, -e.score, e.parent, e.child))

        evidence_lines = []
        for e in selected_edges[:max_items]:
            sentence = self._normalize_text(e.source_sentence)
            line = f"[Hop {e.hop}] {e.parent} --{e.relation}--> {e.child} | {sentence}"
            evidence_lines.append(line)

        return evidence_lines

    def _verify_with_llm(self, claim: str, evidence: List[str]) -> Tuple[str, str]:
        formatted_evidence = "\n".join(evidence)

        if dataset == "mocheg":
            prompt = f"""You are a strict fact-checking assistant.
Based ONLY on the provided context, classify the claim into one of these three categories:
1. supported (The context explicitly confirms the claim)
2. refuted (The context explicitly contradicts the claim)
3. NEI (Not Enough Information in the context to verify)

Context:
{formatted_evidence}

Claim: {claim}

Instructions: Answer ONLY with the label (supported, refuted, or NEI). Do not explain. Do not write your thinking.

Final Answer:"""
        else:
            prompt = f"""You are a strict fact-checking assistant.
Based ONLY on the provided context, classify the claim into one of these categories:
1. supported (The context explicitly confirms the claim)
2. refuted (The context explicitly contradicts the claim)

Context:
{formatted_evidence}

Claim: {claim}

Instructions: Answer ONLY with the label (supported or refuted). Do not explain.

Final Answer:"""

        response = self.ollama.generate(
            prompt,
            temperature=0.3,
            max_tokens=200
        )

        response = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()
        response_upper = response.upper()

        if "SUPPORTED" in response_upper and "REFUTED" not in response_upper:
            prediction = "SUPPORTED"
        elif "REFUTED" in response_upper and "SUPPORTED" not in response_upper:
            prediction = "REFUTED"
        elif "TRUE" in response_upper and "FALSE" not in response_upper:
            prediction = "SUPPORTED"
        elif "FALSE" in response_upper:
            prediction = "REFUTED"
        else:
            prediction = "NEI"

        return prediction, response

    def save_index(self, path: str):
        data = {
            "kg": self.kg,
            "communities": self.communities,
            "partition": self.partition,
            "node_to_sentences": dict(self.node_to_sentences),
            "sentence_to_triples": dict(self.sentence_to_triples),
            "node_embeddings": self.node_embeddings,
            "sentence_embeddings": self.sentence_embeddings,
            "relation_embeddings": self.relation_embeddings,
            "claim_embeddings": self.claim_embeddings,
        }

        with open(path, "wb") as f:
            pickle.dump(data, f)

        print(f"✅ Indice salvato in: {path}")

    def load_index(self, path: str):
        with open(path, "rb") as f:
            data = pickle.load(f)

        self.kg = data["kg"]
        self.communities = data["communities"]
        self.partition = data.get("partition", {})
        self.node_to_sentences = defaultdict(set, data.get("node_to_sentences", {}))
        self.sentence_to_triples = defaultdict(list, data.get("sentence_to_triples", {}))
        self.node_embeddings = data.get("node_embeddings", {})
        self.sentence_embeddings = data.get("sentence_embeddings", {})
        self.relation_embeddings = data.get("relation_embeddings", {})
        self.claim_embeddings = data.get("claim_embeddings", {})

        print(f"✅ Indice caricato da: {path}")


In [168]:
documents = []

print(f"📂 Cerco file .txt in: {DOCUMENTS_DIR}")
txt_files = list(Path(DOCUMENTS_DIR).glob("*.txt"))

if not txt_files:
    print(f"⚠️ Nessun file .txt trovato in {DOCUMENTS_DIR}")
else:
    print(f"📄 Trovati {len(txt_files)} file .txt\n")
    
    for i, txt_file in enumerate(txt_files, 1):
        
        try:
            # Leggi il contenuto del file
            with open(txt_file, 'r', encoding='utf-8') as f:
                contenuto = f.read()
            documents.append(contenuto)
            
        except Exception as e:
            print(f"❌ Errore durante l'inserimento di {txt_file.name}: {e}")        

print("✅ Tutti i documenti sono stati processati\n")

📂 Cerco file .txt in: data/mocheg
📄 Trovati 469 file .txt

✅ Tutti i documenti sono stati processati



In [169]:
INDEX_PATH = WORKING_DIR + "/rag_index_pruned.pkl"

ckgrag = CommunityKGRAG(verbose=False)
os.makedirs(WORKING_DIR, exist_ok=True)

if os.path.exists(INDEX_PATH):
    print("Indice trovato -> carico da disco...")
    ckgrag.load_index(INDEX_PATH)
else:
    print("Indice non trovato -> costruisco...")
    ckgrag.build_knowledge_graph(documents)
    print("detecting_communities")
    ckgrag.detect_communities()
    print("Salvo indice su disco...")
    ckgrag.save_index(INDEX_PATH)
    print("Indice salvato")


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Indice trovato -> carico da disco...
✅ Indice caricato da: dataset_mocheg/rag_index_pruned.pkl


In [170]:
claims = {}

claims_df = pd.read_csv(CLAIMS_CSV)

for index, row in claims_df.iterrows():
    claim = row["Claim"]
    gt = row["cleaned_truthfulness"]

    claims[claim] = gt.upper()

In [171]:
len(claims_df)

100

# 

## Setup

In [175]:
import csv
import os
import time
from itertools import product
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

csv_file_detail = f"{WORKING_DIR}/communitykg_results_detailed_hyper_beta-negativo.csv"
csv_file_summary = f"{WORKING_DIR}/communitykg_results_summary_hyper_beta-negativo.csv"

# =========================
# BUILD GRID
# =========================
BASE_VERIFY_PARAMS = {
    "delta": 0.5,
    "top_k": 10,
    "t_max": 100,
    "use_llm": True
}

ALPHA_VALUES = [0.25]
BETA_VALUES = [0.3]
TAU_VALUES = [0.05]
MAX_RETRIES = 100

param_grid = []
for a, b, t in product(ALPHA_VALUES, BETA_VALUES, TAU_VALUES):
    g = round(1 - a, 2)
    if g >= 0 and g<=1:
        param_grid.append({"alpha": a, "beta": b, "gamma": g, "tau": t})

headers_detail = ["combo_index", "alpha", "beta", "gamma", "tau", "claim_index", "original_claim", "ground_truth", "prediction", "full_llm_response", "processing_time_sec", "num_chars"]
headers_summary = ["combo_index", "alpha", "beta", "gamma", "tau", "num_claims", "num_valid_predictions", "num_errors", "accuracy", "f1_macro", "confusion_matrix_raw"]

claim_items = list(claims.items())
total_claims = len(claim_items)

In [176]:
os.makedirs(os.path.dirname(csv_file_detail), exist_ok=True)
os.makedirs(os.path.dirname(csv_file_summary), exist_ok=True)

max_claim_processed = {}

if os.path.exists(csv_file_detail):
    with open(csv_file_detail, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                c_idx = int(row["combo_index"])
                claim_idx = int(row["claim_index"])
                if c_idx not in max_claim_processed or claim_idx > max_claim_processed[c_idx]:
                    max_claim_processed[c_idx] = claim_idx
            except ValueError:
                pass
    print("🔄 Checkpoint CSV trovato. Progresso calcolato.")
else:
    with open(csv_file_detail, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=headers_detail)
        writer.writeheader()

    with open(csv_file_summary, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=headers_summary)
        writer.writeheader()

    print("🚀 Inizio hyperparameter analysis (nuovo CSV)")


🚀 Inizio hyperparameter analysis (nuovo CSV)


## RUN

In [ ]:
# =========================
# OUTER LOOP ON HYPERPARAMS
# =========================
for combo_index, combo in enumerate(param_grid):
    alpha = combo["alpha"]
    beta = combo["beta"]
    gamma = combo["gamma"]
    tau = combo["tau"]

    VERIFY_PARAMS = BASE_VERIFY_PARAMS.copy()
    VERIFY_PARAMS.update({
        "alpha": alpha,
        "beta": beta,
        "gamma": gamma,
        "tau": tau
    })

    # Decidiamo da dove iniziare per questa specifica combo_index
    if combo_index in max_claim_processed:
        local_start_index = max_claim_processed[combo_index] + 1
    else:
        local_start_index = 0

    # Se tutti i claim di questa combinazione sono stati processati, la saltiamo
    if local_start_index >= total_claims:
        print(f"⏩ Config {combo_index}/{len(param_grid)-1} già completata. Passo alla successiva.")
        continue

    print("=" * 80)
    print(f"⚙️ Config {combo_index}/{len(param_grid)-1} | alpha={alpha}, beta={beta}, gamma={gamma}, tau={tau}")
    print(f"Inizio da claim {local_start_index}")

    # Inner loop
    for index, (claim, gt) in enumerate(claim_items[local_start_index:], local_start_index):
        prediction = "ERROR"
        llm_response = ""
        evidence = None
        last_error = None
        
        start_time = time.perf_counter()
    
        for attempt in range(MAX_RETRIES):
            try:
                result = ckgrag.verify_claim(claim, **VERIFY_PARAMS)
                
                prediction = result["prediction"]
                llm_response = result["llm_response"]
                evidence = result.get("evidence", None)
    
                # print("evidence:", evidence)
                # print(f"\nClaim {index}: {claim}")
                # print("llm_response:", llm_response)
                # print(f"Verdict: {prediction} (GT: {gt})\n")
                
                break
    
            except Exception as e:
                last_error = str(e)
                print(f"❌ Config {combo_index} | Claim {index} | tentativo {attempt + 1}: {last_error[:80]}")
    
                if attempt == MAX_RETRIES - 1:
                    prediction = "ERROR"
                    llm_response = last_error

        if isinstance(evidence, list):
            context_length = sum(len(str(item)) for item in evidence)
        elif evidence:
            context_length = len(str(evidence))
        else:
            context_length = 0
        processing_time_sec = round(time.perf_counter() - start_time, 4)

        row_dict = {
            "combo_index": combo_index,
            "alpha": alpha,
            "beta": beta,
            "gamma": gamma,
            "tau": tau,
            "claim_index": index,
            "original_claim": claim,
            "ground_truth": gt,
            "prediction": prediction,
            "full_llm_response": llm_response,
            "processing_time_sec": processing_time_sec,
            "num_chars": context_length
        }

        # Appendi subito la riga al CSV in modo che il progresso sia salvato per il futuro check
        with open(csv_file_detail, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=headers_detail)
            writer.writerow(row_dict)

    # --- CALCOLO METRICHE A FINE CONFIGURAZIONE ---
    # Per calcolare le metriche di QUESTA combo, rileggiamo il CSV per prendere TUTTI i claim processati (anche in run passate)
    y_true_current = []
    y_pred_current = []
    num_errors = 0
    num_claims_current = 0

    with open(csv_file_detail, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if int(row["combo_index"]) == combo_index:
                num_claims_current += 1
                if row["prediction"] == "ERROR":
                    num_errors += 1
                else:
                    y_true_current.append(row["ground_truth"].strip())
                    y_pred_current.append(row["prediction"].strip())

    try:
        if len(y_true_current) > 0 and len(y_true_current) == len(y_pred_current):
            acc = accuracy_score(y_true_current, y_pred_current)
            f1 = f1_score(y_true_current, y_pred_current, average="macro", zero_division=0)
            conf_matrix_list = confusion_matrix(y_true_current, y_pred_current).tolist()
        else:
            acc, f1, conf_matrix_list = 0, 0, []
    except Exception as e:
        print(f"⚠️ Errore nel calcolo metriche: {e}")
        acc, f1, conf_matrix_list = 0, 0, []

    # Salva riassunto di fine configurazione solo se è appena stata completata in questo giro
    summary_dict = {
        "combo_index": combo_index,
        "alpha": alpha,
        "beta": beta,
        "gamma": gamma,
        "tau": tau,
        "num_claims": num_claims_current,
        "num_valid_predictions": len(y_pred_current),
        "num_errors": num_errors,
        "accuracy": acc,
        "f1_macro": f1,
        "confusion_matrix_raw": str(conf_matrix_list)
    }

    with open(csv_file_summary, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=headers_summary)
        writer.writerow(summary_dict)

    print("-" * 60)
    print(f"✅ Config completata | Accuracy={acc:.4f} | F1={f1:.4f} | Errors={num_errors}")

print("✅ Hyperparameter analysis completata")


⚙️ Config 0/149 | alpha=1, beta=0, gamma=0, tau=0.05
Inizio da claim 0
